# **DATA REVIEW & CLEANSING**



*   Fixing Missing Value
*   Fixing Rows & Columns


*   Fixing datatypes
*   Fixing Invalid Values


*   Filtering Data
*   Downloading clean Data







### **Importing Required Libraries & Dataset**

In [ ]:
import pandas as pd

# Source- Kaggle, Dataset- Lending club loan dataset
# Official link -

path='/content/drive/MyDrive/loan.csv'

data=pd.read_csv(path)
data.head()

/tmp/ipykernel_1894/3190918308.py:8: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv(path)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,1077501,1296599,5000,5000,4975.0,36 months,10.65%,162.87,B,B2,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
1,1077430,1314167,2500,2500,2500.0,60 months,15.27%,59.83,C,C4,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
2,1077175,1313524,2400,2400,2400.0,36 months,15.96%,84.33,C,C5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
3,1076863,1277178,10000,10000,10000.0,36 months,13.49%,339.31,C,C1,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
4,1075358,1311748,3000,3000,3000.0,60 months,12.69%,67.79,B,B5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN


In [ ]:
data.tail()

,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,...,verification_status,issue_d,loan_status,purpose,zip_code,addr_state,dti,pub_rec_bankruptcies,issuemonth,empl_length
39712,92174,2500,2500,1075.0,36,8.07,78.42,A,A4,FiSite Research,...,Not Verified,Jul-07,Fully Paid,home_improvement,802xx,CO,11.33,0.0,Jul-2021,4
39713,90607,8500,8500,875.0,36,10.28,275.38,C,C1,"Squarewave Solutions, Ltd.",...,Not Verified,Jul-07,Fully Paid,credit_card,274xx,NC,6.40,0.0,Jul-2021,3
39714,90390,5000,5000,1325.0,36,8.07,156.84,A,A4,Unknown,...,Not Verified,Jul-07,Fully Paid,debt_consolidation,017xx,MA,2.30,0.0,Jul-2021,0.5
39715,89243,5000,5000,650.0,36,7.43,155.38,A,A2,Unknown,...,Not Verified,Jul-07,Fully Paid,other,208xx,MD,3.72,0.0,Jul-2021,0.5
39716,86999,7500,7500,800.0,36,13.75,255.43,E,E2,Evergreen Center,...,Not Verified,Jun-07,Fully Paid,debt_consolidation,027xx,MA,14.29,0.0,Jun-2021,0.5


In [ ]:
data.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Data columns (total 111 columns):
 #    Column                          Dtype  
---   ------                          -----  
 0    id                              int64  
 1    member_id                       int64  
 2    loan_amnt                       int64  
 3    funded_amnt                     int64  
 4    funded_amnt_inv                 float64
 5    term                            object 
 6    int_rate                        object 
 7    installment                     float64
 8    grade                           object 
 9    sub_grade                       object 
 10   emp_title                       object 
 11   emp_length                      object 
 12   home_ownership                  object 
 13   annual_inc                      float64
 14   verification_status             object 
 15   issue_d                         object 
 16   loan_status                     object 
 17   pymnt_plan

In [ ]:
# Review the null columns %
round(data.isnull().mean()*100,2)

,0
id,0.0
member_id,0.0
loan_amnt,0.0
funded_amnt,0.0
funded_amnt_inv,0.0
...,...
tax_liens,0.1
tot_hi_cred_lim,100.0
total_bal_ex_mort,100.0
total_bc_limit,100.0


In [ ]:
data['desc'].head()

,desc
0,Borrower added on 12/22/11 > I need to upgra...
1,Borrower added on 12/22/11 > I plan to use t...
2,NaN
3,Borrower added on 12/21/11 > to pay for prop...
4,Borrower added on 12/21/11 > I plan on combi...




*   Upon observation, several columns contain 100% missing/null data,rendring then redundant.
*   55 Columns have 100% null values,columns having >75% missing values can be dropped.
*   The desc column can also be dropped since this data is very verbose
*   We can drop this irrelevent columns to streamline the data model.

In [ ]:
columns_to_drop=data.columns[data.isnull().mean()>0.75]
data.shape

data.drop(columns_to_drop,axis=1,inplace=True)
data.shape
data.drop('desc',axis=1,inplace=True)
data.shape



(39717, 54)

##**Post-issuance behavioral features need to be excluded from the features set to prevent Lookahead Bias(Data Leakage.)**


*   **Production Feasibility & Cold Start Problem:** Behavioral metrics are unavailable at the time of credit evaluation for first-time applicants or new borrowers. Relying on post-sanction variables renders the model unusable for real-time loan underwriting.
*   Hence, we can drop such columns which contains customer behavior.














In [ ]:
customer_behavior_columns=['delinq_2yrs','earliest_cr_line', 'inq_last_6mths', 'open_acc', 'pub_rec','revol_bal',
                         'revol_util', 'total_acc', 'out_prncp', 'out_prncp_inv','total_pymnt', 'total_pymnt_inv',
                         'total_rec_prncp', 'total_rec_int','total_rec_late_fee', 'recoveries', 'collection_recovery_fee'
                         ,'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d','mths_since_last_delinq','id','url',
                         'title']
data = data.drop(customer_behavior_columns, axis=1)
data.shape

(39717, 30)

In [ ]:
data.shape

(39717, 30)

## **Fixing Datatype**

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   member_id                   39717 non-null  int64  
 1   loan_amnt                   39717 non-null  int64  
 2   funded_amnt                 39717 non-null  int64  
 3   funded_amnt_inv             39717 non-null  float64
 4   term                        39717 non-null  object 
 5   int_rate                    39717 non-null  object 
 6   installment                 39717 non-null  float64
 7   grade                       39717 non-null  object 
 8   sub_grade                   39717 non-null  object 
 9   emp_title                   37258 non-null  object 
 10  emp_length                  38642 non-null  object 
 11  home_ownership              39717 non-null  object 
 12  annual_inc                  39717 non-null  float64
 13  verification_status         397

In [ ]:
data.nunique().sort_values()

,0
application_type,1
acc_now_delinq,1
chargeoff_within_12_mths,1
delinq_amnt,1
initial_list_status,1
collections_12_mths_ex_med,1
policy_code,1
tax_liens,1
pymnt_plan,1
term,2




*   **Upon observation, there are columns with low variance.**
*   **Columns with low variance can be dropped to reduce dimensionality.**
*   **We drop low-cardinality/low-variance features because they provide negligible information, increase computational cost, and can lead to overfitting without improving accuracy.**



In [ ]:
for col in data.columns:
  if data[col].nunique()==1:
    data.drop(columns=[col],inplace=True)

In [ ]:
data.nunique().sort_values()

,0
term,2
loan_status,3
verification_status,3
pub_rec_bankruptcies,3
home_ownership,5
grade,7
emp_length,11
purpose,14
sub_grade,35
addr_state,50


In [ ]:
data.shape

(39717, 21)

In [ ]:
data['term'].head(15)

,term
0,36 months
1,60 months
2,36 months
3,36 months
4,60 months
5,36 months
6,60 months
7,36 months
8,60 months
9,60 months


In [ ]:
# In column 'term' extra words need to remove
data['term']=data['term'].str.replace("months",'')
data['term'].value_counts()

,count
term,
36,29096
60,10621


In [ ]:
data['loan_status'].value_counts()

,count
loan_status,
Fully Paid,32950
Charged Off,5627
Current,1140


In [ ]:
data['verification_status'].value_counts()

,count
verification_status,
Not Verified,16921
Verified,12809
Source Verified,9987




*   **The loan_status & verification_status features can be retained as-is.**




In [ ]:
data['pub_rec_bankruptcies'].isna()


,pub_rec_bankruptcies
0,False
1,False
2,False
3,False
4,False
...,...
39712,True
39713,True
39714,True
39715,True


In [ ]:
# Filling null values with 0 assuming no previous public record of bankrupt.
data['pub_rec_bankruptcies'].fillna(0)


,pub_rec_bankruptcies
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
39712,0.0
39713,0.0
39714,0.0
39715,0.0


In [ ]:
data['home_ownership'].value_counts()

,count
home_ownership,
RENT,18899
MORTGAGE,17659
OWN,3058
OTHER,98
NONE,3


In [ ]:
data['home_ownership'].isnull().sum()

np.int64(0)

In [ ]:
data['grade'].value_counts()

,count
grade,
B,12020
A,10085
C,8098
D,5307
E,2842
F,1049
G,316


In [ ]:
data['grade'].isnull().sum()

np.int64(0)



*   **The home_ownership and grade columns can be retained as is.**




In [ ]:
data['emp_length'].value_counts()

,count
emp_length,
10+ years,8879
< 1 year,4583
2 years,4388
3 years,4095
4 years,3436
5 years,3282
1 year,3240
6 years,2229
7 years,1773


In [ ]:
# Change the datatype of employe lenth and fixing the cell by removing extra words.

data['emp_length']=data['emp_length'].str.replace('years','')
data['emp_length']=data['emp_length'].str.replace("+",'')
data['emp_length']=data['emp_length'].str.replace("< 1",'0.5')
data['emp_length']=data['emp_length'].str.replace("year",'')



In [ ]:
data['emp_length'].value_counts()

,count
emp_length,
10,8879
0.5,4583
2,4388
3,4095
4,3436
5,3282
1,3240
6,2229
7,1773


In [ ]:
data['purpose'].value_counts()

,count
purpose,
debt_consolidation,18641
credit_card,5130
other,3993
home_improvement,2976
major_purchase,2187
small_business,1828
car,1549
wedding,947
medical,693


In [ ]:
data['purpose'].isnull().sum()

np.int64(0)

In [ ]:
data['sub_grade'].value_counts()

,count
sub_grade,
B3,2917
A4,2886
A5,2742
B5,2704
B4,2512
C1,2136
B2,2057
C2,2011
B1,1830


In [ ]:
data['sub_grade'].isnull().sum()

np.int64(0)

In [ ]:
data['addr_state'].isnull().sum()

np.int64(0)



*   **NO action required for add_state, sub_grade &purpose as they contain clean, valid and predictive data.**




In [ ]:
data['issue_d'].value_counts()

,count
issue_d,
Dec-11,2260
Nov-11,2223
Oct-11,2114
Sep-11,2063
Aug-11,1928
Jul-11,1870
Jun-11,1827
May-11,1689
Apr-11,1562


In [ ]:
# Adding year in issue_id in the format
data['issuemonth'] = data['issue_d'].str[:3]+'-2021'

In [ ]:
data.issuemonth.value_counts()

,count
issuemonth,
Dec-2021,4433
Nov-2021,4167
Oct-2021,3934
Sep-2021,3648
Aug-2021,3518
Jul-2021,3476
Jun-2021,3279
May-2021,2999
Apr-2021,2834


In [ ]:
data['int_rate'].value_counts()

,count
int_rate,
10.99%,956
13.49%,826
11.49%,825
7.51%,787
7.88%,725
...,...
17.34%,1
16.71%,1
16.15%,1


In [ ]:
# removing the extra word %
data['int_rate']=data['int_rate'].str.replace("%",'')

In [ ]:
# Change datatype
data['int_rate']=data['int_rate'].astype('float64')

In [ ]:
data['int_rate'].isnull().sum()

np.int64(0)

In [ ]:
data['zip_code'].head()

,zip_code
0,860xx
1,309xx
2,606xx
3,917xx
4,972xx


In [ ]:
data['zip_code'].isnull().sum()

np.int64(0)

In [ ]:
data['loan_amnt'].isnull().sum()

np.int64(0)

In [ ]:
data['loan_amnt'].value_counts()

,count
loan_amnt,
10000,2833
12000,2334
5000,2051
6000,1908
15000,1895
...,...
18125,1
23050,1
11175,1


In [ ]:
data['funded_amnt'].isnull().sum()

np.int64(0)

In [ ]:
data['funded_amnt'].value_counts()

,count
funded_amnt,
10000,2741
12000,2244
5000,2040
6000,1898
15000,1784
...,...
950,1
700,1
725,1


In [ ]:
data['dti'].isnull().sum()

np.int64(0)

In [ ]:
data['dti'].value_counts()

,count
dti,
0.00,183
12.00,51
18.00,45
19.20,40
13.20,39
...,...
29.46,1
27.25,1
25.24,1


In [ ]:
data['dti'].head()

,dti
0,27.65
1,1.00
2,8.72
3,20.00
4,17.94


In [ ]:
data.isnull().sum()

,0
member_id,0
loan_amnt,0
funded_amnt,0
funded_amnt_inv,0
term,0
int_rate,0
installment,0
grade,0
sub_grade,0
emp_title,2459


In [ ]:
data['emp_title']=data['emp_title'].fillna('Unknown')


In [ ]:
data['empl_length']=data['emp_length'].fillna(0)

In [ ]:
data['emp_length'].head()

,emp_length
0,10
1,0.5
2,10
3,10
4,1




*   **Rest all feilds are not required to change**List item




In [ ]:
# checking the clean dataset
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   member_id             39717 non-null  int64  
 1   loan_amnt             39717 non-null  int64  
 2   funded_amnt           39717 non-null  int64  
 3   funded_amnt_inv       39717 non-null  float64
 4   term                  39717 non-null  object 
 5   int_rate              39717 non-null  float64
 6   installment           39717 non-null  float64
 7   grade                 39717 non-null  object 
 8   sub_grade             39717 non-null  object 
 9   emp_title             39717 non-null  object 
 10  emp_length            38642 non-null  object 
 11  home_ownership        39717 non-null  object 
 12  annual_inc            39717 non-null  float64
 13  verification_status   39717 non-null  object 
 14  issue_d               39717 non-null  object 
 15  loan_status        

In [ ]:
data.isnull().sum()

,0
member_id,0
loan_amnt,0
funded_amnt,0
funded_amnt_inv,0
term,0
int_rate,0
installment,0
grade,0
sub_grade,0
emp_title,0


In [ ]:
data['pub_rec_bankruptcies'].head()

,pub_rec_bankruptcies
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


In [ ]:
data['pub_rec_bankruptcies']=data['pub_rec_bankruptcies'].fillna(0.0)

In [ ]:
data['pub_rec_bankruptcies'].isnull().sum()

np.int64(0)

In [ ]:
data['emp_length']=data['emp_length'].fillna("0")

In [ ]:
data['emp_length'].isnull().sum()

np.int64(0)

In [ ]:
data.isnull().sum()

,0
member_id,0
loan_amnt,0
funded_amnt,0
funded_amnt_inv,0
term,0
int_rate,0
installment,0
grade,0
sub_grade,0
emp_title,0


In [ ]:
data.describe()

,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,pub_rec_bankruptcies
count,3.971700e+04,39717.000000,39717.000000,39717.000000,39717.000000,39717.000000,3.971700e+04,39717.000000,39717.000000
mean,8.504636e+05,11219.443815,10947.713196,10397.448868,12.021177,324.561922,6.896893e+04,13.315130,0.042501
std,2.656783e+05,7456.670694,7187.238670,7128.450439,3.724825,208.874874,6.379377e+04,6.678594,0.202603
min,7.069900e+04,500.000000,500.000000,0.000000,5.420000,15.690000,4.000000e+03,0.000000,0.000000
25%,6.667800e+05,5500.000000,5400.000000,5000.000000,9.250000,167.020000,4.040400e+04,8.170000,0.000000
50%,8.508120e+05,10000.000000,9600.000000,8975.000000,11.860000,280.220000,5.900000e+04,13.400000,0.000000
75%,1.047339e+06,15000.000000,15000.000000,14400.000000,14.590000,430.780000,8.230000e+04,18.600000,0.000000
max,1.314167e+06,35000.000000,35000.000000,35000.000000,24.590000,1305.190000,6.000000e+06,29.990000,2.000000


In [ ]:
data['member_id'].head()

,member_id
0,1296599
1,1314167
2,1313524
3,1277178
4,1311748


In [ ]:
# member id should be a object
data['member_id']=data['member_id'].astype(object)

In [ ]:
data['member_id'].head()

,member_id
0,1296599
1,1314167
2,1313524
3,1277178
4,1311748


In [ ]:
data.describe()

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,pub_rec_bankruptcies
count,39717.000000,39717.000000,39717.000000,39717.000000,39717.000000,3.971700e+04,39717.000000,39717.000000
mean,11219.443815,10947.713196,10397.448868,12.021177,324.561922,6.896893e+04,13.315130,0.042501
std,7456.670694,7187.238670,7128.450439,3.724825,208.874874,6.379377e+04,6.678594,0.202603
min,500.000000,500.000000,0.000000,5.420000,15.690000,4.000000e+03,0.000000,0.000000
25%,5500.000000,5400.000000,5000.000000,9.250000,167.020000,4.040400e+04,8.170000,0.000000
50%,10000.000000,9600.000000,8975.000000,11.860000,280.220000,5.900000e+04,13.400000,0.000000
75%,15000.000000,15000.000000,14400.000000,14.590000,430.780000,8.230000e+04,18.600000,0.000000
max,35000.000000,35000.000000,35000.000000,24.590000,1305.190000,6.000000e+06,29.990000,2.000000


In [ ]:
data['funded_amnt_inv'].head()

,funded_amnt_inv
0,4975.0
1,2500.0
2,2400.0
3,10000.0
4,3000.0


## **Downloading the cleaned data**

In [ ]:
from google.colab import files
data.to_csv('lending_club_loan_clean_data.csv')
files.download('lending_club_loan_clean_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>